© 2025 Mobile Perception Systems Lab at TU/e. All rights reserved. Licensed under the MIT License.

## Setup
The initial version of the code (for semantic segmentation) was using the **eomt_base_640.yaml** config file. However, the weights couldn't be loaded from Hugging Face (because they didn't exist). Decision was taken to use the **eomt_large_1024.yaml** config file, which allowed download of the weights (it has nonetheless been noticed that this config file was deleted from the original repository of the code).

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import importlib
import warnings
import torch
import yaml

from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.metrics import average_precision_score
from torch.amp.autocast_mode import autocast
from huggingface_hub import hf_hub_download
from torch.utils.data import DataLoader
from ood_metrics import fpr_at_95_tpr
from lightning import seed_everything
from torch.nn import functional as F
from torchvision import transforms
from pathlib import Path
from PIL import Image

torch.cuda.memory.empty_cache()

seed_everything(0, verbose=False)

device = 0
img_idx = 10  # the index of the image you want to visualize
config_path = "configs\dinov2\cityscapes\semantic\eomt_large_1024.yaml"
data_path = r"C:\Users\edgar\Documents\segmentation_project\code\anomaly_segmentation\ValidationDatasets"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array([plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))])
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping

def apply_colormap(image, mapping):
    """
    Function to apply colors to the predicted image (output of the model) and target of the image, when displaying the results.
    """
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image

C:\Users\edgar\Documents\segmentation_project\code\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load dataset

The initial version of this file was designed to load data from cityscapes. In order to be able to load data from the given datasets (FS_LostFound_full, fs_static, RoadAnomaly, RoadAnomaly21 and RoadObsticle21) we need to add a few things :

In [2]:
# custom class to load the images from the chosen datasets (which have all the same structure)
class AnomalyFolderDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, dataset_name, img_size):
        self.img_dir = Path(data_path) / dataset_name / "images"
        self.lbl_dir = Path(data_path) / dataset_name / "labels_masks"
        self.img_size = img_size

        self.image_paths = sorted(self.img_dir.glob("*.*"))
        self.label_paths = []
        for img_path in self.image_paths:
            lbl_path = self.lbl_dir / (img_path.stem + ".png")
            if lbl_path.exists():
                self.label_paths.append(lbl_path)
            else:
                print(f"Warning: missing label for {img_path.name}")

        self.input_transform = transforms.Compose([
            transforms.Resize(img_size, Image.BILINEAR),
            transforms.ToTensor(),
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize(img_size, Image.NEAREST),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        print(f"Loading image: {img_path}")
        img = Image.open(img_path).convert("RGB")
        print(f"  Original size: {img.size}")
        img = self.input_transform(img)
        print(f"  After transform shape: {img.shape}")
        if img.shape[1] < 10 or img.shape[2] < 10:
            print(f"  WARNING: Image too small! Skipping?")
        lbl_path = self.label_paths[idx]
        lbl = Image.open(lbl_path)
        lbl = self.target_transform(lbl)
        lbl = torch.as_tensor(np.array(lbl), dtype=torch.long)
        return img, lbl

# custom class that returns the dataset under a usable shape by the model
class AnomalyDataWrapper:
    def __init__(self, data_path, dataset_name, img_size, num_classes=19):
        self.img_size = img_size
        self.num_classes = num_classes
        self.dataset = AnomalyFolderDataset(data_path, dataset_name, img_size)

    def val_dataloader(self):
        return DataLoader(self.dataset, batch_size=1, num_workers=0)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


img_size = (1024, 1024)

dataset_name = "fs_static"
data = AnomalyDataWrapper(
    data_path=data_path,
    dataset_name=dataset_name,
    img_size=img_size,
    num_classes=19
)

## Load model

In [ ]:
warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network"}
if "stuff_classes" in config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

## Load pre-trained weights from Hugging Face Hub
The model weights are downloaded from the Hugging Face Hub using the logger name from the config. Make sure you have a working internet connection.

In [ ]:
name = config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")

if name is None:
    warnings.warn("No logger name found in the config. Please specify a model name.")
else:
    try:
        state_dict_path = hf_hub_download(
            repo_id=f"tue-mps/{name}",
            filename="pytorch_model.bin",
        )

        is_dinov3 = "dinov3" in name

        if is_dinov3:
            model_kwargs["ckpt_path"] = state_dict_path
            model_kwargs["delta_weights"] = True

        model = (
            lit_cls(
                img_size=data.img_size,
                num_classes=data.num_classes,
                network=network,
                **model_kwargs,
            )
            .eval()
            .to(device)
        )

        if not is_dinov3:
            state_dict = torch.load(
                state_dict_path, map_location=f"cuda:{device}", weights_only=True
            )
            model.load_state_dict(state_dict, strict=False)

    except RepositoryNotFoundError:
        warnings.warn(
            f"Pre-trained model not found for `{name}`. Please load your own checkpoint."
        )

## Semantic inference (pixel-wise classification)

> This inference method also works when applied to a model trained for panoptic segmentation.

Semantic inference computes per-pixel class scores by combining mask and class predictions:

$$
\sum_i p_i(c) \cdot m_i[h, w]
$$

Here, $p_i(c)$ is the class probability for class $c$ (excluding "no object"), and $m_i[h, w]$ is the sigmoid-normalized mask value for query $i$ at pixel $(h, w)$. The final class is selected by taking the argmax over classes.  
  
*This inference method was originally introduced in MaskFormer.*

In [ ]:
IGNORE_INDEX = 255

torch.cuda.memory.empty_cache()

def infer_semantic(img, target):
    """
    Function to compute inference on a single image
    Returns predicted class map, resized target map, and logits
    """
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        # Get model's required input size (ensures divisibility)
        h, w = img.shape[-2:]
        new_h, new_w = model.scale_img_size_semantic((h, w))
        print(f"Resizing from ({h},{w}) to ({new_h},{new_w})")

        # Resize image to the model's required size
        img_resized = F.interpolate(
            img.unsqueeze(0), size=(new_h, new_w), mode='bilinear', align_corners=False
        ).squeeze(0)
        batched_img = img_resized.unsqueeze(0).to(device)

        # Forward pass
        mask_logits_per_layer, class_logits_per_layer = model(batched_img)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], data.img_size, mode="bilinear"
        )

        # Convert to per-pixel logits
        logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        preds = logits[0].argmax(0).cpu()
        pred_array = preds.numpy()

        # Process target: resize to the same size as the image after scaling
        # target is a tensor of shape (H, W) (no batch or channel dims)
        target_resized = F.interpolate(
            target.float().unsqueeze(0).unsqueeze(0),
            size=(new_h, new_w),
            mode='nearest'
        ).squeeze().long()
        target_array = target_resized.cpu().numpy()

    return pred_array, target_array, logits

def calculate_msp(logits):
    """
    Standard Maximum Softmax Probability (MSP) for OoD detection.
    """
    # Subtract max for numerical stability
    logits_shifted = logits - np.max(logits, axis=0, keepdims=True)
    exp_logits = np.exp(logits_shifted)
    softmax = exp_logits / np.sum(exp_logits, axis=0, keepdims=True)
    return np.max(softmax, axis=0)

def calculate_entropy(logits):
    """
    Calculates the entropy from the logits. Definition established from the one defined in the SegmentMeIfYouCan project.
    """
    softmax_output = np.exp(logits) / np.sum(np.exp(logits), axis=0, keepdims=True)
    log_probs = np.log(softmax_output)# + 1e-10)
    return np.sum(-softmax_output * log_probs, axis=0)

def calculate_max_logit(logits):
    """
    Calculates the maximum logit. Definition from Scaling OoD Detection for Real-World Settings
    """
    return -np.max(logits, axis=0)

def calculate_metrics(scores, ood_mask, ind_mask):
    """
    Calculates AUPRC and FPR@TPR95, using the initial implemented method.
    """
    ood_out = scores[ood_mask]
    ind_out = scores[ind_mask]

    ood_label = np.ones(len(ood_out))
    ind_label = np.zeros(len(ind_out))

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((ind_label, ood_label))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    return prc_auc, fpr

def plot_semantic_results(img, pred_array, target_array):
    mapping = create_mapping([pred_array, target_array], IGNORE_INDEX)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title("Image")
    axes[1].imshow(apply_colormap(pred_array, mapping))
    axes[1].set_title("Prediction")
    axes[2].imshow(apply_colormap(target_array, mapping))
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

msp_list = []
entropy_list = []
max_logit_list = []
target_list = []
pred_list = []
ood_gts_list = []

for index in range(len(data.val_dataloader().dataset)-10):
    img, target = data.val_dataloader().dataset[index]
    torch.cuda.memory.empty_cache()
    img, target = data.val_dataloader().dataset[index]
    pred_array, target_array, logits = infer_semantic(img, target)

    logits_np = logits[0].cpu().numpy()

    msp_list.append(calculate_msp(logits_np))
    entropy_list.append(calculate_entropy(logits_np))
    max_logit_list.append(calculate_max_logit(logits_np))

    # Keep the original target for visualisation
    original_target = target_array.copy()

    # Dataset‑specific label correction
    corrected_target = target_array.copy()

    if dataset_name == "RoadAnomaly":
        corrected_target = np.where(corrected_target == 2, 1, corrected_target)

    elif dataset_name == "LostAndFound":
        corrected_target = np.where(corrected_target == 0, 255, corrected_target)
        corrected_target = np.where(corrected_target == 1, 0, corrected_target)
        corrected_target = np.where((corrected_target > 1) & (corrected_target < 201), 1, corrected_target)

    # Skip images with no anomaly
    if 1 not in np.unique(corrected_target):
        continue

    ood_gts_list.append(corrected_target)
    target_list.append(original_target)
    pred_list.append(pred_array)

# Now build masks from corrected ground truth
ood_gts_array = np.array(ood_gts_list)
ood_mask = (ood_gts_array == 1)
ind_mask = (ood_gts_array == 0)
ignore_mask = (ood_gts_array == 255)

ood_mask = ood_mask & ~ignore_mask
ind_mask = ind_mask & ~ignore_mask

print(f"for {dataset_name} dataset:") 

MSP_scores = calculate_metrics(np.array(msp_list), ood_mask, ind_mask)
print(f"MSP:\n \t->  AuPRC : {MSP_scores[0]}\n \t->  FPR95 : {MSP_scores[1]}")

entropy_scores = calculate_metrics(np.array(entropy_list), ood_mask, ind_mask)
print(f"MaxEntropy:\n \t->  AuPRC : {entropy_scores[0]}\n \t->  FPR95 : {entropy_scores[1]}")

max_logit_scores = calculate_metrics(np.array(max_logit_list), ood_mask, ind_mask)
print(f"MaxLogit:\n \t->  AuPRC : {max_logit_scores[0]}\n \t->  FPR95 : {max_logit_scores[1]}")

plot_semantic_results(img, pred_array, target_array)